# Notebook 2c -- Few-step distillation (~45 min)

This track notebook builds on Notebook 1; Notebook 2a helps but is not
required.  It distills the spiral diffusion teacher into a **1-step
student**, compares quality against cost, and isolates the mechanism that
limits 1-step maps.

Three markers:
- ✏️ marks an exercise
- 📦 marks provided code or context (just read/run)
- ⭐ marks optional extra material

Three terms we will use throughout:

- **NFE** is the number of neural-network forward evaluations per generated
  sample.
- The **PF-ODE** (probability-flow ODE) is the deterministic ODE with the
  same marginals $p_t$ as the diffusion SDE (see the boxed definition in the
  Diffusion models section of the companion post, *Tutorial: Generative
  models as transport*).
- The **score** is $s(x, t) = \nabla_x \log p_t(x)$.

**Convention** (as in Notebook 1 §2): $t$ runs from 0 (pure noise) to 1
(data), the direction of generation.  DDPM-style papers run time the other
way, so their $t$ is our $1 - t$.  Endpoints are $x$ (data) and
$z \sim N(0, I)$ (noise).

In [ ]:
import time
from functools import partial
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np
import flax.nnx as nnx
import optax

import matplotlib.pyplot as plt
from tqdm import tqdm

import iaifi_gm as gm  # helper package

# Repo-relative paths (works from the repo root or from notebooks/).
REPO = Path.cwd() if (Path.cwd() / "checkpoints").exists() else Path.cwd().parent
CKPT_DIR = REPO / "checkpoints"

rngs = nnx.Rngs(0)
gm.plotting.use_style()
gm.plotting.device_report()

## 1. The problem & the teacher

In Notebook 1, good spiral samples took a PF-ODE solve of roughly 50-128
steps -- one NFE per step.  For a 2D toy that is milliseconds; for an
image model it is the deployment bottleneck, because sampling cost scales as
NFE times the cost of one forward pass, and the forward pass is a large
network.

This notebook takes the most direct route, **distillation**: train a cheap
*student* network to reproduce the expensive teacher in a single forward pass.

📦 We load the shipped spiral diffusion teacher (the same checkpoint 2a uses
for its comparison; training script: `scripts/train_spiral_diffusion.py`).
If you trained your own model in Notebook 1, you can point `TEACHER_CKPT` at
your saved checkpoint instead -- e.g. `checkpoints/my_spiral_diffusion.msgpack`
if you ran Notebook 1's save cell.

In [ ]:
TEACHER_CKPT = CKPT_DIR / "spiral_diffusion.msgpack"
assert TEACHER_CKPT.exists(), (
    "Teacher checkpoint not found -- run `python "
    "scripts/train_spiral_diffusion.py` from the repo root (~15 s) to create it."
)

teacher = gm.models.TimeMLP(
    dim=2, hidden=128, depth=3, time_dim=32, rngs=nnx.Rngs(params=0)
)
teacher = gm.checkpoints.load(teacher, TEACHER_CKPT)
print(f"loaded teacher from {TEACHER_CKPT}")

📦 **Teacher ODE-solve helper.**  This is exactly Notebook 1 Exercise 4's
PF-ODE Euler sampler, batched and jit'd.

The update it implements, on the trig VP schedule
$\alpha_t = \sin(\pi t / 2)$, $\sigma_t = \cos(\pi t / 2)$:

$$ x \leftarrow x + \Delta t \, \big( a(t)\, x + b(t)\, \varepsilon_\theta(x, t) \big),
   \qquad a(t) = \tfrac{\pi}{2} \cot\big(\tfrac{\pi t}{2}\big),
   \quad  b(t) = -\tfrac{\pi}{2} \big/ \sin\big(\tfrac{\pi t}{2}\big), $$

so the ε-to-velocity conversion is baked in: $\varepsilon_\theta$ predicts the
noise, the score is $s_\theta = -\varepsilon_\theta / \sigma_t$, and the PF-ODE
velocity is the linear combination above.  The grid is clipped to
$t \in [t_{\min}, 1 - t_{\min}]$ with $t_{\min} = 10^{-2}$: the coefficients
blow up like $1/t$ at the noise end, so without the clip the integration fails.
`kind="velocity"` covers flow-matching
teachers, whose output is the velocity directly and whose grid is the full $[0, 1]$.

In [ ]:
T_MIN = 1e-2
TEACHER_STEPS = 50  # comfortable default

alpha = lambda t: jnp.sin(jnp.pi * t / 2)
sigma = lambda t: jnp.cos(jnp.pi * t / 2)


@partial(nnx.jit, static_argnums=(4, 5))
def ode_solve_span(model, x, t0, t1, n_steps, kind="eps"):
    """Fixed-step Euler integration of the PF-ODE from t0 to t1.

    x: (B, d) state at time t0  ->  (B, d) state at time t1.
    kind="eps": model predicts noise (diffusion; trig-VP conversion built in);
    kind="velocity": model output is dx/dt directly (flow matching).
    """
    ts = jnp.linspace(t0, t1, n_steps + 1)

    def euler_step(x, i):
        t, dt = ts[i], ts[i + 1] - ts[i]
        out = model(x, jnp.full(x.shape[0], t))
        if kind == "eps":
            a = (jnp.pi / 2) / jnp.tan(jnp.pi * t / 2)
            b = -(jnp.pi / 2) / jnp.sin(jnp.pi * t / 2)
            v = a * x + b * out
        else:
            v = out
        return x + dt * v, None

    x, _ = jax.lax.scan(euler_step, x, jnp.arange(n_steps))
    return x


def ode_solve(model, z, n_steps, kind="eps"):
    """Full generation z -> x; grid [T_MIN, 1 - T_MIN] for eps, [0, 1] for velocity."""
    t0, t1 = (T_MIN, 1 - T_MIN) if kind == "eps" else (0.0, 1.0)
    return ode_solve_span(model, z, t0, t1, n_steps, kind)


# Smoke test: the teacher still draws the spiral.
z_smoke = jax.random.normal(jax.random.key(0), (2048, 2))
x_smoke = ode_solve(teacher, z_smoke, TEACHER_STEPS)
ax = gm.plotting.scatter2d(x_smoke)
ax.set_title(f"teacher, {TEACHER_STEPS}-step PF-ODE Euler ({TEACHER_STEPS} NFE/sample)")

The PF-ODE defines a *deterministic* map
$z \mapsto x$: fix the noise draw, and the sample is fully determined.  A
student network $f_\theta$ can therefore regress that map directly,

$$ f_\theta(z) \approx \mathrm{ODESolve}(z), $$

and regression is the right tool *because* the target is a single-valued
function of $z$, not a distribution.  If $f_\theta$ succeeds, sampling costs
1 NFE: draw $z$, one forward pass.

## 2. Distillation dataset

We pay the teacher's cost **once**, offline: generate a dataset of
$(z, x = \mathrm{ODESolve}(z))$ pairs, then never call the teacher at
inference time again.

Three practical notes before the exercise:

- **Generation cost** is the number of pairs times the teacher's NFE.  That
  is the one-time price we pay to make inference cheap.
- **The pair count also sets student quality.**  With too few pairs the
  student overfits a scatter of points instead of the map.  We use 16k pairs
  for a 2-in/2-out MLP, plus a held-out set for error analysis.
- **We generate in chunks.**  One giant batch would be fine in 2D, but
  chunking would be needed for a real (memory-bound) computation.

### ✏️ Exercise 1 -- generate and cache teacher pairs

Fill in `make_pairs`: for each provided key, draw a chunk of noise
$z \sim N(0, I)$, push it through the provided `ode_solve` teacher sampler,
and stack everything into two flat arrays.  Pass `kind` through to `ode_solve`
(we reuse this function for a flow-matching teacher in §3).

In [ ]:
N_PAIRS = 16384
CHUNK = 4096
keys_data = jax.random.split(jax.random.key(1), N_PAIRS // CHUNK)  # one key per chunk
keys_val = jax.random.split(jax.random.key(2), 1)  # held-out chunk for §3


def make_pairs(model, keys, chunk_size, n_steps, kind="eps"):
    """Generate (z, x) distillation pairs with the teacher.

    keys: (K,) PRNG keys, one per chunk -> z: (K * chunk_size, 2),
    x: (K * chunk_size, 2) with x[i] = ODESolve(z[i]).
    """
    # SOLUTION
    z_chunks, x_chunks = [], []
    for key in keys:
        z = jax.random.normal(key, (chunk_size, 2))
        z_chunks.append(z)
        x_chunks.append(ode_solve(model, z, n_steps, kind))
    return jnp.concatenate(z_chunks), jnp.concatenate(x_chunks)
    # END SOLUTION

In [ ]:
# 📦 Shape check -- run before the full generation.
z_tiny, x_tiny = make_pairs(teacher, jax.random.split(jax.random.key(9), 2), 128, 8)
assert z_tiny.shape == (256, 2), f"z shape {z_tiny.shape}, expected (256, 2)"
assert x_tiny.shape == (256, 2), f"x shape {x_tiny.shape}, expected (256, 2)"
assert bool(jnp.all(jnp.isfinite(x_tiny))), "teacher outputs are not finite"
assert float(jnp.abs(x_tiny).max()) < 10.0, "teacher outputs look wrong (|x| too large)"
print("shapes OK")

In [ ]:
# 📦 Full dataset + held-out validation pairs.
t0 = time.perf_counter()
z_data, x_data = make_pairs(teacher, keys_data, CHUNK, TEACHER_STEPS)
z_val, x_val = make_pairs(teacher, keys_val, CHUNK, TEACHER_STEPS)
n_total = z_data.shape[0] + z_val.shape[0]
print(f"{n_total} pairs in {time.perf_counter() - t0:.1f} s "
      f"(one-time teacher bill: {n_total * TEACHER_STEPS:,} NFE)")

## 3. Train the 1-step student

The student is the same MLP class as the teacher (`gm.models.TimeMLP`) with
its time input pinned to 0 -- a plain map $\hat x = f_\theta(z)$, no time
dependence -- but with a deliberately **smaller size** (32 hidden units,
depth 2).  The handicap is deliberate: on a 2D toy a teacher-sized student
simply learns the map perfectly (swap in `hidden=128, depth=3` to check) and
there is nothing left to study.  At image scale that is unrealistic to expect.
Capacity is always scarce relative to the map the network must represent.
The small size puts us in that interesting regime on a laptop.

### ✏️ Exercise 2 -- the distillation loss

Plain mean-squared-error regression on the cached pairs:

$$ L(\theta) = \mathbb{E}_{(z, x) \sim \text{pairs}}
   \big\| f_\theta(z) - x \big\|^2 . $$

Fill in the loss body; the train step below reuses Notebook 1's training
loop, and batches are indexed out of the cached arrays.

In [ ]:
def distill_loss(student, z_batch, x_batch):
    """MSE between student prediction and teacher output.

    z_batch: (B, 2), x_batch: (B, 2) -> scalar.
    The student is called as student(z_batch, 0.0) -- time pinned to 0.
    """
    # SOLUTION
    pred = student(z_batch, 0.0)
    return jnp.mean((pred - x_batch) ** 2)
    # END SOLUTION

In [ ]:
# 📦 Shape check -- run before training.
STUDENT_KW = dict(dim=2, hidden=32, depth=2, time_dim=32)  # the small student trunk

student_test = gm.models.TimeMLP(**STUDENT_KW, rngs=nnx.Rngs(params=99))
loss_test = distill_loss(student_test, z_data[:8], x_data[:8])
assert jnp.shape(loss_test) == (), f"loss must be a scalar, got {jnp.shape(loss_test)}"
assert bool(jnp.isfinite(loss_test)), "loss is not finite"
pred_test = student_test(z_data[:8], 0.0)
assert pred_test.shape == (8, 2), f"prediction shape {pred_test.shape}, expected (8, 2)"
print(f"untrained student loss: {float(loss_test):.3f}")

📦 Training follows the same train-step pattern as Notebook 1 and takes
about 5 s.  `fit_student` builds a fresh
student so that we can rerun the identical recipe on a different teacher's
pairs at the end of this section.

In [ ]:
def fit_student(z_pairs, x_pairs, n_steps=4000, batch=256, lr=1e-3, seed=0):
    """Train a fresh 1-step student on cached (z, x) pairs; returns (student, losses)."""
    student = gm.models.TimeMLP(**STUDENT_KW, rngs=nnx.Rngs(params=seed))
    optimizer = nnx.Optimizer(
        student, optax.adam(optax.cosine_decay_schedule(lr, n_steps)), wrt=nnx.Param
    )

    @nnx.jit
    def train_step(student, optimizer, key):
        idx = jax.random.randint(key, (batch,), 0, z_pairs.shape[0])
        loss, grads = nnx.value_and_grad(distill_loss)(
            student, z_pairs[idx], x_pairs[idx]
        )
        optimizer.update(grads=grads, model=student)
        return loss

    losses = np.full(n_steps, np.nan)
    keys = jax.random.split(jax.random.key(seed + 100), n_steps)
    for i in tqdm(range(n_steps), desc="distill"):
        losses[i] = train_step(student, optimizer, keys[i])
    return student, losses


student, losses = fit_student(z_data, x_data)
print(f"final distillation loss (mean of last 100): {losses[-100:].mean():.4f}")

📦 **Evaluation: quality vs NFE.**  We put three sample sets side by side
and report a few numbers per sampler:

- The **energy distance** to the target (the same metric as 2a) is a
  distance between sample sets that vanishes exactly when the distributions
  coincide.  Always read it against the printed target-vs-target noise floor.
- The **haze fraction** is the fraction of samples landing where the target
  density is lowest (below the target's own 5th percentile).  Energy distance
  is a global metric and quite forgiving of a thin mist of stray points; the
  haze fraction is a magnifying glass for exactly the artifact we are about
  to discuss.

In [ ]:
@nnx.jit
def student_sample(student, z):
    return student(z, 0.0)


def best_of_3(fn, *args):
    fn(*args).block_until_ready()  # warm-up / compile
    times = []
    for _ in range(3):
        t0 = time.perf_counter()
        fn(*args).block_until_ready()
        times.append(time.perf_counter() - t0)
    return min(times)


N_EVAL = 2048
key_eval, key_ref, key_ref2 = jax.random.split(jax.random.key(3), 3)
z_eval = jax.random.normal(key_eval, (N_EVAL, 2))
x_target = gm.targets.sample_spiral(key_ref, N_EVAL)
ed_floor = gm.metrics.energy_distance(gm.targets.sample_spiral(key_ref2, N_EVAL), x_target)

ld_threshold = jnp.percentile(gm.targets.spiral_log_density(x_target), 5.0)


def haze_fraction(x):
    """Fraction of samples below the target's 5th-percentile log-density."""
    return float(jnp.mean(gm.targets.spiral_log_density(x) < ld_threshold))


x_teacher = ode_solve(teacher, z_eval, TEACHER_STEPS)
x_student = student_sample(student, z_eval)
ed_teacher = gm.metrics.energy_distance(x_teacher, x_target)
ed_student = gm.metrics.energy_distance(x_student, x_target)
t_teacher = best_of_3(ode_solve, teacher, z_eval, TEACHER_STEPS)
t_student = best_of_3(student_sample, student, z_eval)

print(f"energy distance, target-vs-target noise floor: {ed_floor:.4f}")
print(f"{'':12s} {'NFE':>5s} {'energy dist':>12s} {'haze frac':>10s} {'ms / batch of 2048':>20s}")
print(f"{'target':12s} {'-':>5s} {'-':>12s} {haze_fraction(x_target):10.3f} {'-':>20s}")
print(f"{'teacher':12s} {TEACHER_STEPS:5d} {ed_teacher:12.4f} {haze_fraction(x_teacher):10.3f} {1e3 * t_teacher:20.1f}")
print(f"{'student':12s} {1:5d} {ed_student:12.4f} {haze_fraction(x_student):10.3f} {1e3 * t_student:20.1f}")

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, samples, title in [
    (axes[0], x_target, "target"),
    (axes[1], x_teacher, f"teacher ({TEACHER_STEPS} NFE)"),
    (axes[2], x_student, "student (1 NFE)"),
]:
    gm.plotting.scatter2d(samples, ax=ax)
    ax.set_title(title)

The student is 50× cheaper in NFE -- and cheaper still in wall-clock time,
since each evaluation is also a smaller network -- and it clearly draws *a*
spiral.  Look closely, though: a haze of stray points fills the gap between
the arms, something the teacher barely produces, and the student's haze
fraction is roughly twice the teacher's.

The mechanism is the **stiffness of the teacher's flow map**.  The map
$z \mapsto x$ is nearly discontinuous across *basin boundaries* in $z$-space:
two neighboring noise draws can land on different spiral arms, far apart in
$x$-space.  A smooth, finite-capacity student cannot reproduce a
near-discontinuity.  It must interpolate across the boundary, and the
interpolated outputs land in the gaps *between* the arms -- exactly where the
haze lives.

📦 We can see this directly: color the $z$-plane by which blob of the
(unswirled) mixture the teacher transports each point to, and overlay the
held-out points where the student's error is largest.

In [ ]:
n_grid = 200
grid = jnp.linspace(-3, 3, n_grid)
zz1, zz2 = jnp.meshgrid(grid, grid)
z_grid = jnp.stack([zz1, zz2], axis=-1).reshape(-1, 2)
x_dest = ode_solve(teacher, z_grid, TEACHER_STEPS)
# Destination arm: undo the swirl, check which mixture blob the endpoint is in.
arm = gm.targets.swirl(x_dest, sign=-1.0)[:, 0] > 0

err_val = jnp.linalg.norm(student_sample(student, z_val) - x_val, axis=-1)
worst = jnp.argsort(-err_val)[:300]

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.imshow(
    np.asarray(arm).reshape(n_grid, n_grid),
    origin="lower", extent=(-3, 3, -3, 3), cmap="coolwarm", alpha=0.45,
)
ax.scatter(*np.asarray(z_val[worst]).T, s=6, c="k", label="student's 300 worst z's")
ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal")
ax.legend(loc="upper left")
ax.set_title("z-plane, colored by destination arm under the teacher map")
print(f"median student error on held-out pairs: {float(jnp.median(err_val)):.3f}")
print(f"mean error of the 300 worst:            {float(err_val[worst].mean()):.3f}")

The teacher's basins spiral around each other, and the student's largest
errors concentrate precisely on the basin boundaries.

This mechanism also tells you what *would* help: over a **shorter time
interval** the teacher's sub-map is far less stiff, because a half-time map
moves points less far and neighboring $z$'s have not yet separated onto
different arms.  That is why §4's few-step sub-maps are so much easier to regress.

📦 **Teacher dependence.**  Flow matching (2a) trains a velocity field
on *straight* interpolation paths; its ODE trajectories are much straighter
than the diffusion PF-ODE's, and its $z \mapsto x$ map is correspondingly
milder.  The cell below loads the pre-trained spiral FM teacher and reruns the
*identical* pipeline on its pairs -- with 32 solver steps instead of 50, which
suffice because the straighter FM ODE is converged by then (cf. 2a's NFE sweep).

In [ ]:
FM_CKPT = CKPT_DIR / "spiral_fm.msgpack"

if FM_CKPT.exists():
    teacher_fm = gm.models.TimeMLP(
        dim=2, hidden=128, depth=3, time_dim=32, rngs=nnx.Rngs(params=0)
    )
    teacher_fm = gm.checkpoints.load(teacher_fm, FM_CKPT)
    z_fm, x_fm = make_pairs(teacher_fm, keys_data, CHUNK, 32, kind="velocity")
    student_fm, losses_fm = fit_student(z_fm, x_fm, seed=1)

    x_student_fm = student_sample(student_fm, z_eval)
    ed_student_fm = gm.metrics.energy_distance(x_student_fm, x_target)
    print(f"final distillation loss:  diffusion teacher {losses[-100:].mean():.4f}"
          f"  |  FM teacher {losses_fm[-100:].mean():.4f}")
    print(f"1-step student energy distance:  from diffusion {ed_student:.4f}"
          f"  |  from FM {ed_student_fm:.4f}   (floor {ed_floor:.4f})")
    print(f"1-step student haze fraction:    from diffusion {haze_fraction(x_student):.3f}"
          f"  |  from FM {haze_fraction(x_student_fm):.3f}")

    fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
    axes[0].plot(losses, label="diffusion teacher", alpha=0.8)
    axes[0].plot(losses_fm, label="FM teacher", alpha=0.8)
    axes[0].set_yscale("log")
    axes[0].set_xlabel("step"); axes[0].set_ylabel("distillation loss")
    axes[0].legend()
    gm.plotting.scatter2d(x_student, ax=axes[1])
    axes[1].set_title("student of the diffusion teacher")
    gm.plotting.scatter2d(x_student_fm, ax=axes[2])
    axes[2].set_title("student of the FM teacher")
else:
    print("FM checkpoint not found -- skipping the teacher comparison.")
    print("To create it: python scripts/train_spiral_fm.py  (~15 s)")

Same student, same recipe, same pair budget -- yet the FM teacher distills to
a lower regression loss and a cleaner 1-step student. The straighter the
teacher's ODE, the easier its endpoint map is to fit.

## 4. Beyond naive regression

**Few-step students.**  Split $[0, 1]$ into $K$ segments and learn the
teacher's map over each segment -- one student per segment, or one student
conditioned on the segment's start time.  Each sub-map moves points less far
and is easier to *regress*: §3's mechanism in reverse.  Turning easier
regression into better end-to-end samples takes two more ideas, and the ⭐
stretch lets you run into both.  First, errors **compound** across steps, so
each stage should train on the *student's* own intermediate outputs -- the
teacher's ODE can be restarted from any state, not just from its own
trajectories.  Second, **where you cut matters**: with the VP schedule the
arms separate late, so uniform cuts in $t$ leave the last sub-map holding
most of the stiffness.

**Consistency models** take a more elegant route to the same place: learn a
single function $g_\theta(x_t, t)$ that maps *any* point on a PF-ODE
trajectory to that trajectory's endpoint at $t = 1$.  Self-consistency --
adjacent points on one trajectory must map to the same endpoint -- is the
training signal, and $g(\cdot, t \approx 0)$ is then a 1-step sampler.
Two distinct terms:

- In consistency **distillation**, the targets come from *single teacher
  steps* between adjacent times -- there is no precomputed $(z, x)$ dataset
  at all.
- In consistency **training**, there is no teacher: the target comes from
  the model itself at an adjacent time, a noisier learning signal.

📦 The demo below is a minimal consistency *distillation* on the spiral
([Song et al., 2023](https://arxiv.org/abs/2303.01469)).  The ingredients
are a boundary-anchored parametrization $g_\theta(x, t) = x + (1 - t)\,
\mathrm{net}_\theta(x, t)$ (so that $g = x$ at $t = 1$ by construction), a
random grid time $t_n$, a point $x_{t_n} = \alpha_{t_n} x + \sigma_{t_n} z$
on the noising path, **one** teacher Euler step to $t_{n+1}$, and the loss
$\| g_\theta(x_{t_n}, t_n) - \mathrm{sg}\, g_\theta(x_{t_{n+1}}, t_{n+1}) \|^2$
with sg = stop-gradient.

In [ ]:
N_CD_GRID = 32
ts_cd = jnp.linspace(T_MIN, 1 - T_MIN, N_CD_GRID + 1)
CD_BATCH, CD_STEPS = 256, 6000

cd_net = gm.models.TimeMLP(**STUDENT_KW, rngs=nnx.Rngs(params=5))  # same trunk as the student (the longer schedule is standard for CD)
cd_opt = nnx.Optimizer(
    cd_net, optax.adam(optax.cosine_decay_schedule(1e-3, CD_STEPS)), wrt=nnx.Param
)


def g_fn(net, x, t):
    """Boundary-anchored consistency function; g(x, 1) = x by construction."""
    t = jnp.broadcast_to(t, x.shape[:-1])
    return x + (1 - t)[..., None] * net(x, t)


@nnx.jit
def cd_step(net, optimizer, teacher, key):
    key_x, key_z, key_n = jax.random.split(key, 3)
    x = gm.targets.sample_spiral(key_x, CD_BATCH)
    z = jax.random.normal(key_z, x.shape)
    n = jax.random.randint(key_n, (CD_BATCH,), 0, N_CD_GRID)
    t_lo, t_hi = ts_cd[n], ts_cd[n + 1]
    x_lo = alpha(t_lo)[:, None] * x + sigma(t_lo)[:, None] * z  # a point on p_{t_lo}
    # One teacher PF-ODE Euler step t_lo -> t_hi (per-sample time):
    a = (jnp.pi / 2) / jnp.tan(jnp.pi * t_lo / 2)
    b = -(jnp.pi / 2) / jnp.sin(jnp.pi * t_lo / 2)
    v = a[:, None] * x_lo + b[:, None] * teacher(x_lo, t_lo)
    x_hi = x_lo + (t_hi - t_lo)[:, None] * v
    target = jax.lax.stop_gradient(g_fn(net, x_hi, t_hi))

    def loss_fn(net):
        return jnp.mean((g_fn(net, x_lo, t_lo) - target) ** 2)

    loss, grads = nnx.value_and_grad(loss_fn)(net)
    optimizer.update(grads=grads, model=net)
    return loss

cd_losses = np.full(CD_STEPS, np.nan)
cd_keys = jax.random.split(jax.random.key(6), CD_STEPS)
for i in tqdm(range(CD_STEPS), desc="consistency"):
    cd_losses[i] = cd_step(cd_net, cd_opt, teacher, cd_keys[i])

x_cd = g_fn(cd_net, z_eval, jnp.full(N_EVAL, T_MIN))  # 1-step samples
ed_cd = gm.metrics.energy_distance(x_cd, x_target)
print(f"consistency 1-step energy distance: {ed_cd:.4f} "
      f"(regression student: {ed_student:.4f}, floor: {ed_floor:.4f})")
ax = gm.plotting.scatter2d(x_cd)
ax.set_title("consistency distillation, 1 NFE")

This sampler does not quite match our
pair-trained student on this toy, but it was built with **no cached
dataset**: the teacher was only ever queried for single adjacent-time steps.

A few pointers:

- **Trajectory distillation.**  Progressive distillation
  ([Salimans & Ho, 2022](https://arxiv.org/abs/2202.00512)) repeatedly
  halves the step count -- learn one step that matches the teacher's two,
  then recurse -- and consistency models generalize the endpoint-map idea.
- **Distribution matching.**  DMD (distribution matching distillation;
  [Yin et al., 2023](https://arxiv.org/abs/2311.18828)) and
  adversarial distillation drop the pairwise regression entirely and only
  ask that the student's output *distribution* match the teacher's,
  sidestepping §3's stiffness problem at the cost of a harder training
  setup.
- **Training-free efficiency.**  Quantization, caching, and better solvers
  need no distillation at all; see the lectures and the TinyML course linked
  from the README.

## Wrap-up

The PF-ODE makes generation a **deterministic map**, so a student can **regress** that map and sample in
1 NFE.  What limits 1-step quality is the map's **stiffness** across basin
boundaries.  Shorter sub-maps are easier to regress,
and **few-step** students and **consistency models** convert that into
quality at 2-4 NFE at scale, where the 1-step map is hopeless.
Meanwhile **straighter teachers** (FM, 2a) distill better from the start.

## ⭐ Stretch 1 -- few-step students vs just using fewer solver steps

The lazy way to cut NFE is to run the *teacher* with fewer Euler steps.  The
distillation way is a $K$-step student: split $[t_{\min}, 1 - t_{\min}]$ at
$t_0 < t_1 < \dots < t_K$ and train one small net per segment.  We build the
stages **sequentially** to fight compounding: stage $k$ trains on the
*student's* own intermediate outputs, with targets from the teacher restarted
there.  Sampling costs $K$ NFE.

✏️ Fill in (a) the stage-$k$ teacher targets and (b) the composed sampler;
everything reuses `fit_student` unchanged (each stage is again a plain
regression with the time input pinned to 0).

In [ ]:
def fit_fewstep(K, n_pairs=8192, seed=0):
    """Distill a K-step student (one net per segment, trained sequentially).

    Returns a sampler z: (B, 2) -> x: (B, 2) costing K NFE.
    """
    ts_seg = jnp.linspace(T_MIN, 1 - T_MIN, K + 1)
    x_in = jax.random.normal(jax.random.key(7), (n_pairs, 2))
    nets = []
    for k in range(K):
        # (a) Stage-k targets: evolve the CURRENT inputs x_in with the teacher
        #     from ts_seg[k] to ts_seg[k + 1] (use TEACHER_STEPS // K Euler steps).
        #     x_out: (n_pairs, 2)
        # SOLUTION
        x_out = ode_solve_span(teacher, x_in, ts_seg[k], ts_seg[k + 1], TEACHER_STEPS // K)
        # END SOLUTION
        net, seg_losses = fit_student(x_in, x_out, seed=seed + 10 * K + k)
        print(f"  stage {k}: t in [{float(ts_seg[k]):.2f}, {float(ts_seg[k + 1]):.2f}], "
              f"final loss {seg_losses[-100:].mean():.4f}")
        nets.append(net)
        x_in = student_sample(net, x_in)  # next stage sees the student's outputs

    def sampler(z):
        """Compose the K stages: z -> x in K forward passes."""
        x = z
        # (b) apply each stage net in order (student_sample(net, x))
        # SOLUTION
        for net in nets:
            x = student_sample(net, x)
        # END SOLUTION
        return x

    return sampler

In [ ]:
# 📦 Check: cheap run before the full comparison.
sampler_pf = fit_fewstep(2, n_pairs=256)
z_pf = jax.random.normal(jax.random.key(8), (256, 2))
x_pf = sampler_pf(z_pf)
assert x_pf.shape == (256, 2), f"sampler output shape {x_pf.shape}, expected (256, 2)"
assert bool(jnp.all(jnp.isfinite(x_pf))), "sampler output is not finite"
assert float(jnp.abs(x_pf - z_pf).max()) > 1e-3, "sampler output equals its input"
print("few-step sampler pre-flight OK")

In [ ]:
# Baseline: the raw teacher at few Euler steps (NB1's naive way to cut NFE).
euler_ed = {n: float(gm.metrics.energy_distance(ode_solve(teacher, z_eval, n), x_target))
            for n in (1, 2, 4, 8, 16, TEACHER_STEPS)}
student_ed = {1: float(ed_student)}
student_haze = {1: haze_fraction(x_student)}
for K in (2, 4):
    sampler_k = fit_fewstep(K)
    x_k = sampler_k(z_eval)
    student_ed[K] = float(gm.metrics.energy_distance(x_k, x_target))
    student_haze[K] = haze_fraction(x_k)

fig, ax = plt.subplots(figsize=(5.5, 3.8))
ns = sorted(euler_ed)
ax.plot(ns, [euler_ed[n] for n in ns], "o-", label="teacher, fewer Euler steps")
ks = sorted(student_ed)
ax.plot(ks, [student_ed[k] for k in ks], "s-", label="distilled K-step student")
ax.axhline(float(ed_floor), color="k", ls="--", lw=1, label="sampling noise floor")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xticks(ns); ax.set_xticklabels(ns)
ax.set_xlabel("NFE"); ax.set_ylabel("energy distance")
ax.legend()
ax.set_title("quality vs NFE: distillation vs naive step-cutting")
print("teacher-Euler ED:", {n: round(v, 4) for n, v in euler_ed.items()})
print("student ED:      ", {k: round(v, 4) for k, v in student_ed.items()})
print("student haze:    ", {k: round(v, 3) for k, v in student_haze.items()})

- **Distillation wins the low-NFE regime by orders of magnitude.**  The
  teacher at 1-4 Euler steps is off by a huge margin -- the very first step
  at the noise end is the stiffest, Notebook 1's $1/t$ blow-up -- while the
  distilled students sit near teacher quality at the same NFE.
- **On this toy, $K = 2, 4$ do *not* beat $K = 1$.**  The stage losses
  printed above show that the sub-maps *are* individually easier (the early
  stage fits ~5× better than the full map).  But uniform-in-$t$ cuts leave
  the last stage holding most of the stiffness, since the arms only separate
  late in the VP schedule, and small per-stage errors compound -- together
  this cancels the gain.  At image scale, where the 1-step map is hopeless
  rather than merely hazy, the same trade lands firmly in favor of a few
  steps, and the two failure channels just measured (cut placement,
  compounding) are the things to tune.

## ⭐ Stretch 2 -- distill a fashion-MNIST teacher

This stretch runs the same pipeline at image scale, using the provided
fashion-MNIST diffusion checkpoint (`gm.data` + `gm.checkpoints`, with a
`SmallUNet` student).

In [ ]:
FMNIST_CKPT = CKPT_DIR / "fmnist_diffusion.msgpack"

if not FMNIST_CKPT.exists():
    print("fashion-MNIST diffusion checkpoint not found -- skipping this stretch.")
    print("(Train it with scripts/train_fmnist_diffusion.py on a GPU;")
    print(" nothing below depends on it.)")
else:
    print("Expected runtime on a laptop CPU: ~5-10 min (pair generation dominates).")
    N_IMG_PAIRS, IMG_TEACHER_STEPS, IMG_CHUNK = 256, 50, 64
    # Constructor args spelled out to match the checkpoint (same as NB1 §6).
    unet_teacher = gm.models.SmallUNet(
        channels=(32, 64, 128), in_channels=1, time_dim=128, rngs=nnx.Rngs(params=0)
    )
    unet_teacher = gm.checkpoints.load(unet_teacher, FMNIST_CKPT)

    # Pair generation: ode_solve works unchanged (broadcasting handles images).
    # Its sampling grid starts at T_MIN = 1e-2 -- matching NB1 §6 and the
    # training script's sample() (training clips at 1e-3, a superset).
    img_keys = jax.random.split(jax.random.key(11), N_IMG_PAIRS // IMG_CHUNK)
    z_chunks, x_chunks = [], []
    for key in tqdm(img_keys, desc="teacher pairs"):
        z_img = jax.random.normal(key, (IMG_CHUNK, 28, 28, 1))
        z_chunks.append(z_img)
        x_chunks.append(ode_solve(unet_teacher, z_img, IMG_TEACHER_STEPS))
    z_imgs, x_imgs = jnp.concatenate(z_chunks), jnp.concatenate(x_chunks)

    unet_student = gm.models.SmallUNet(
        channels=(32, 64, 128), in_channels=1, time_dim=128, rngs=nnx.Rngs(params=1)
    )
    unet_opt = nnx.Optimizer(unet_student, optax.adam(2e-4), wrt=nnx.Param)

    @nnx.jit
    def unet_step(student, optimizer, key):
        idx = jax.random.randint(key, (32,), 0, z_imgs.shape[0])

        def loss_fn(student):
            return jnp.mean((student(z_imgs[idx], 0.0) - x_imgs[idx]) ** 2)

        loss, grads = nnx.value_and_grad(loss_fn)(student)
        optimizer.update(grads=grads, model=student)
        return loss

    for key in tqdm(jax.random.split(jax.random.key(12), 1000), desc="distill unet"):
        unet_step(unet_student, unet_opt, key)

    z_show = jax.random.normal(jax.random.key(13), (16, 28, 28, 1))
    fig_t = gm.plotting.image_grid(ode_solve(unet_teacher, z_show, IMG_TEACHER_STEPS))
    fig_t.suptitle(f"teacher ({IMG_TEACHER_STEPS} NFE)")
    fig_s = gm.plotting.image_grid(unet_student(z_show, 0.0))
    fig_s.suptitle("student (1 NFE) -- expect blur at this pair budget")